# Task 2 - Unsupervised Domain Adaptation

PACS Source-only, DAN, DANN, and CDAN under the specified transductive protocol.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration and protocol

This notebook uses PACS sources Photo/Art Painting/Cartoon and unlabeled Sketch. Do not inspect target-label metrics until `final_evaluation` at the end.

The next cell fixes the PACS protocol, source/target batch composition, optimization budget, main MMD setting, and the controlled-study values. It produces no metrics.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd().resolve()
if not (ROOT / "ATML-PA1.pdf").exists():
    ROOT = ROOT.parent


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()

from torchvision.models import resnet18, ResNet18_Weights

CONFIG = {
    "pacs_root": ROOT / "data" / "PACS",
    "results_dir": ROOT / "task2" / "results",
    "batch_size_per_domain": 8,
    "target_batch_size": 24,
    "epochs": 30,
    "patience": 5,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "mmd_lambda": 1.0,
    "study_lambdas": [0.1, 1.0, 10.0],
    "plot": True,
    "print_metrics": True,
}
SOURCES = ["photo", "art_painting", "cartoon"]
TARGET = "sketch"
CLASSES = ["dog", "elephant", "giraffe", "guitar", "horse", "house", "person"]
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)


## 2. PACS indexing, fixed source splits, transforms, and frozen BatchNorm statistics

The next cell indexes PACS, writes the shared deterministic source split for Task 3, sets the prescribed augmentations, and defines the frozen-BatchNorm ResNet-18.

In [ ]:
# Expected PACS layout: PACS/<domain>/<class>/*.jpg. The JSON split is shared with Task 3.
def index_pacs(root, domains):
    rows = []
    for domain in domains:
        for label, name in enumerate(CLASSES):
            for path in sorted((root / domain / name).glob("*")):
                rows.append({"path": str(path), "domain": domain, "label": label})
    return pd.DataFrame(rows)


all_source = index_pacs(CONFIG["pacs_root"], SOURCES)
target_index = index_pacs(CONFIG["pacs_root"], [TARGET])
rng = np.random.default_rng(SEED)
split_rows = []
# Split within each source-domain/class group so both class balance and domain identity are preserved.
for (domain, label), group in all_source.groupby(["domain", "label"]):
    ids = group.index.to_numpy().copy()
    rng.shuffle(ids)
    cut = int(0.8 * len(ids))
    split_rows.extend(
        {"index": int(i), "split": "train" if n < cut else "val"} for n, i in enumerate(ids)
    )
splits = pd.DataFrame(split_rows).set_index("index")
all_source = all_source.join(splits)
save_json(
    all_source.reset_index().to_dict("records"),
    ROOT / "common" / "splits" / "pacs_sketch_seed6304.json",
)
weights = ResNet18_Weights.IMAGENET1K_V1
train_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)


class PACSDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        return self.transform(Image.open(row.path).convert("RGB")), int(row.label), row.domain


# Keep pretrained running moments fixed; gamma and beta remain trainable parameters.
def freeze_bn_statistics(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def make_model():
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 7)
    return model.to(DEVICE)


## 3. Shared losses and training utilities

The next cell defines MMD, gradient reversal, the domain discriminator, 512-dimensional feature extraction, domain-balanced loaders, and accuracy/macro-F1 evaluation helpers.

In [ ]:
# RBF MMD compares source and target feature distributions without target labels.
def gaussian_mmd(source, target):
    joined = torch.cat([source, target])
    distances = torch.cdist(joined, joined).pow(2)
    median = distances[distances > 0].median().detach().clamp_min(1e-6)
    kernels = sum(torch.exp(-distances / (2 * (scale * median))) for scale in (0.5, 1.0, 2.0))
    n = len(source)
    m = len(target)
    return kernels[:n, :n].mean() + kernels[n:, n:].mean() - 2 * kernels[:n, n:].mean()


class GradientReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


def grl(x, alpha):
    return GradientReverse.apply(x, alpha)


class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2)
        )

    def forward(self, x, alpha):
        return self.net(grl(x, alpha))


# The feature tensor has shape [batch, 512], immediately before the seven-class classifier.
def features_and_logits(model, x):
    # Mirror ResNet's forward pass, while retaining the 512-D vector before fc.
    h = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    h = model.layer1(h)
    h = model.layer2(h)
    h = model.layer3(h)
    h = model.layer4(h)
    feature = torch.flatten(model.avgpool(h), 1)
    return feature, model.fc(feature)


def domain_loaders(frame, split, transform):
    return {
        d: DataLoader(
            PACSDataset(frame[(frame.domain == d) & (frame.split == split)], transform),
            batch_size=CONFIG["batch_size_per_domain"],
            shuffle=True,
            drop_last=True,
        )
        for d in SOURCES
    }


def evaluate(model, frame, transform=eval_tf):
    model.eval()
    freeze_bn_statistics(model)
    ys = []
    ps = []
    for x, y, _ in DataLoader(PACSDataset(frame, transform), batch_size=64):
        with torch.no_grad():
            _, z = features_and_logits(model, x.to(DEVICE))
            ps.extend(z.argmax(1).cpu())
            ys.extend(y)
    return {
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": f1_score(ys, ps, average="macro"),
        "y": np.array(ys),
        "pred": np.array(ps),
    }


## 4. Source-only, DAN, DANN, and CDAN

Each method uses domain-balanced source batches plus 24 unlabeled target images. Checkpoint selection uses only mean source-validation macro-F1.

The next cell trains Source-only, DAN, DANN, and CDAN with a common pipeline. It records classification/alignment losses and uses only mean source-validation macro-F1 to select checkpoints.

In [ ]:
# Target labels are never read here; they cannot influence gradients or checkpoint selection.
def train_method(method, alignment_strength=1.0):
    set_seed()
    model = make_model()
    discriminator = None
    disc_dim = 512 if method == "dann" else 512 * 7
    # Gradient reversal trains the discriminator normally while reversing only its gradient into the backbone.
    if method in {"dann", "cdan"}:
        discriminator = DomainDiscriminator(disc_dim).to(DEVICE)
    optimizer = torch.optim.AdamW(
        list(model.parameters())
        + ([] if discriminator is None else list(discriminator.parameters())),
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    source_iters = {
        d: iter(loader) for d, loader in domain_loaders(all_source, "train", train_tf).items()
    }
    target_loader = DataLoader(
        PACSDataset(target_index.assign(split="target"), train_tf),
        batch_size=CONFIG["target_batch_size"],
        shuffle=True,
        drop_last=True,
    )
    target_iter = iter(target_loader)
    best, best_state, stale, history = -1, None, 0, []
    for epoch in range(CONFIG["epochs"]):
        model.train()
        freeze_bn_statistics(model)
        total_cls = total_align = 0
        steps = min(
            len(loader) for loader in domain_loaders(all_source, "train", train_tf).values()
        )
        for step in range(steps):
            source_batches = []
            for d in SOURCES:
                try:
                    batch = next(source_iters[d])
                except StopIteration:
                    source_iters[d] = iter(domain_loaders(all_source, "train", train_tf)[d])
                    batch = next(source_iters[d])
                source_batches.append(batch)
            try:
                target_batch = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                target_batch = next(target_iter)
            sx = torch.cat([b[0] for b in source_batches]).to(DEVICE)
            sy = torch.cat([b[1] for b in source_batches]).to(DEVICE)
            tx = target_batch[0].to(DEVICE)
            sf, sz = features_and_logits(model, sx)
            tf, _ = features_and_logits(model, tx)
            cls = F.cross_entropy(sz, sy)
            align = torch.zeros((), device=DEVICE)
            if method == "dan":
                align = alignment_strength * gaussian_mmd(sf, tf)
            if method in {"dann", "cdan"}:
                progress = (epoch * steps + step) / (CONFIG["epochs"] * steps)
                alpha = 2 / (1 + math.exp(-10 * progress)) - 1
                if method == "cdan":
                    sf_in = torch.einsum("bi,bj->bij", sf, F.softmax(sz, 1)).flatten(1)
                    tf_logits = features_and_logits(model, tx)[1]
                    tf_in = torch.einsum("bi,bj->bij", tf, F.softmax(tf_logits, 1)).flatten(1)
                else:
                    sf_in, tf_in = sf, tf
                domain_logits = discriminator(torch.cat([sf_in, tf_in]), alpha)
                domain_labels = torch.cat(
                    [torch.zeros(len(sf), dtype=torch.long), torch.ones(len(tf), dtype=torch.long)]
                ).to(DEVICE)
                align = F.cross_entropy(domain_logits, domain_labels)
            optimizer.zero_grad()
            (cls + align).backward()
            optimizer.step()
            total_cls += cls.item()
            total_align += align.item()
        val_scores = [
            evaluate(model, all_source[(all_source.domain == d) & (all_source.split == "val")])[
                "macro_f1"
            ]
            for d in SOURCES
        ]
        score = float(np.mean(val_scores))
        history.append(
            {
                "epoch": epoch + 1,
                "classification_loss": total_cls / steps,
                "alignment_or_domain_loss": total_align / steps,
                "mean_source_val_f1": score,
            }
        )
        if score > best:
            best, best_state, stale = (
                score,
                {
                    "model": copy.deepcopy(model.state_dict()),
                    "disc": (
                        None if discriminator is None else copy.deepcopy(discriminator.state_dict())
                    ),
                },
                0,
            )
        else:
            stale += 1
        if stale >= CONFIG["patience"]:
            break
    model.load_state_dict(best_state["model"])
    return model, history


models_by_method = {}
histories = {}
for method in ["source_only", "dan", "dann", "cdan"]:
    models_by_method[method], histories[method] = train_method(method, CONFIG["mmd_lambda"])


## 5. Final evaluation, domain separability, per-class changes, and controlled MMD study

The next cell performs final source/target metrics, domain separability, and the preconfigured MMD-strength study. It saves CSV tables and an optional training-loss plot.

In [ ]:
def domain_separability(model):
    frames = [
        all_source[(all_source.domain == d) & (all_source.split == "val")].assign(domain_id=i)
        for i, d in enumerate(SOURCES)
    ]
    source = pd.concat(frames)
    target = target_index.assign(domain_id=3)
    n = min(len(source), len(target))
    balanced = pd.concat([source.sample(n, random_state=SEED), target.sample(n, random_state=SEED)])
    feats = []
    domains = []
    for x, _, d in DataLoader(PACSDataset(balanced, eval_tf), batch_size=64):
        with torch.no_grad():
            feats.append(features_and_logits(model, x.to(DEVICE))[0].cpu())
            domains.extend((d == TARGET).long().numpy())
    ids = np.random.default_rng(SEED).permutation(len(domains))
    cut = int(0.7 * len(ids))
    clf = LogisticRegression(C=1, class_weight="balanced", max_iter=2000).fit(
        torch.cat(feats)[ids[:cut]], np.array(domains)[ids[:cut]]
    )
    return clf.score(torch.cat(feats)[ids[cut:]], np.array(domains)[ids[cut:]])


rows = []
for method, model in models_by_method.items():
    source_scores = {
        d: evaluate(model, all_source[(all_source.domain == d) & (all_source.split == "val")])
        for d in SOURCES
    }
    target_score = evaluate(model, target_index)  # first and only target-label use: final analysis
    rows.append(
        {
            "method": method,
            **{f"{d}_accuracy": source_scores[d]["accuracy"] for d in SOURCES},
            "mean_source_accuracy": np.mean([x["accuracy"] for x in source_scores.values()]),
            "mean_source_macro_f1": np.mean([x["macro_f1"] for x in source_scores.values()]),
            "target_accuracy": target_score["accuracy"],
            "target_macro_f1": target_score["macro_f1"],
            "domain_separability": domain_separability(model),
        }
    )
frame = pd.DataFrame(rows)
frame["target_accuracy_change_vs_source_only"] = (
    frame.target_accuracy - frame.loc[frame.method == "source_only", "target_accuracy"].iloc[0]
)
if CONFIG["print_metrics"]:
    display(frame)
frame.to_csv(RESULTS / "final_metrics.csv", index=False)
# Controlled study: configurations are fixed; do not choose a setting from Sketch results.
study = []
for value in CONFIG["study_lambdas"]:
    model, _ = train_method("dan", value)
    study.append(
        {
            "lambda_mmd": value,
            "source_macro_f1": np.mean(
                [
                    evaluate(
                        model, all_source[(all_source.domain == d) & (all_source.split == "val")]
                    )["macro_f1"]
                    for d in SOURCES
                ]
            ),
            "target_accuracy_final_analysis_only": evaluate(model, target_index)["accuracy"],
            "domain_separability": domain_separability(model),
        }
    )
pd.DataFrame(study).to_csv(RESULTS / "mmd_strength_study.csv", index=False)
if CONFIG["plot"]:
    for method, history in histories.items():
        plt.plot(
            [x["epoch"] for x in history], [x["classification_loss"] for x in history], label=method
        )
    plt.legend()
    plt.title("Classification-loss curves")
    plt.show()
